In [ ]:
import pandas as pd
import polars as pl
from loguru import logger
from pathlib import Path
import hydra
from functools import reduce
from omegaconf import DictConfig, OmegaConf

import hvplot.pandas
import holoviews as hv
import matplotlib.pyplot as plt

from workspace_dani.data_preprocessing.utils.data_handling import load_parquet_table

hv.extension("bokeh")
#pd.options.plotting.backend = 'holoviews

In [ ]:

project_root_path = Path.cwd().parents[2]  #/ Path("data_preprocessing") / Path("utils")

dataset_path = project_root_path / Path("intermediary_data") / Path("dataset") / Path("sepsis-type-b-cohort") / Path(
    "2024-05-31") / Path("build_dataset_ts.parquet")

diag_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-b/2024-05-30/diagnosis.parquet")
lab_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-b/2024-05-30/lab.parquet")
aper_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-b/2024-05-30/vital_aperiodic.parquet")
per_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-b/2024-05-30/vital_periodic.parquet")



In [ ]:
reference_dataset = pl.scan_parquet(dataset_path)


In [ ]:
ts_tables_paths: list[Path] = [lab_path, diag_path, per_path, aper_path]

In [ ]:
df_list_ts: list[pl.DataFrame] = [
    #pl.scan_parquet(table) for table in  ts_tables_paths
    pl.read_parquet(table) for table in  ts_tables_paths
]

In [ ]:
#df_list_ts[0].head().collect()
df_list_ts[0].head()

In [ ]:
#df_list_ts[1].head().collect()
df_list_ts[1].head()

In [ ]:
df_joined_ts_sample_negative =  df_list_ts[0].join(df_list_ts[1], on=["patientunitstayid", "time_offset"], how="full", coalesce=True, join_nulls=False)
#df_joined_ts_sample_negative.head().collect()
df_joined_ts_sample_negative.head()

In [ ]:
df_joined_ts_sample_negative_with_other = df_joined_ts_sample_negative.join(df_list_ts[2], on=["patientunitstayid", "time_offset"], how="full", coalesce=True, join_nulls=True)
#df_joined_ts_sample_negative_with_other = df_list_ts[2].join(df_list_ts[3], on=["patientunitstayid", "time_offset"], how="full", coalesce=True, join_nulls=True)



In [ ]:
#df_joined_ts_sample_negative_with_other.head().collect()
df_joined_ts_sample_negative_with_other.head()

In [ ]:
hand_joined_df_seq = df_list_ts[0].join(df_list_ts[1], on=["patientunitstayid", "time_offset"], how="full", coalesce=True, join_nulls=True)
hand_joined_df_seq = hand_joined_df_seq.join(df_list_ts[2], on=["patientunitstayid", "time_offset"], how="full", coalesce=True, join_nulls=True)
hand_joined_df_seq = hand_joined_df_seq.join(df_list_ts[3], on=["patientunitstayid", "time_offset"], how="full", coalesce=True, join_nulls=True)


hand_joined_df_seq.head()

In [ ]:
hand_joined_df_branch_01 = df_list_ts[0].join(df_list_ts[1], on=["patientunitstayid", "time_offset"], how="full", coalesce=True, join_nulls=True)
hand_joined_df_branch_23 = df_list_ts[2].join(df_list_ts[3], on=["patientunitstayid", "time_offset"], how="full", coalesce=True, join_nulls=True)

hand_joined_df_branch = hand_joined_df_branch_01.join(hand_joined_df_branch_23, on=["patientunitstayid", "time_offset"], how="full", coalesce=True, join_nulls=True)

In [ ]:
hand_joined_df_branch_01.head()


In [ ]:
hand_joined_df_branch_23.head()

In [ ]:
hand_joined_df_branch.head()

In [ ]:
sum_sub_joins = len(hand_joined_df_branch_01) + len(hand_joined_df_branch_23)

delta_joins = sum_sub_joins - len(hand_joined_df_branch)

delta_joins

## Muss mal merge ausprobieren

In [ ]:
df_joined_ts: pl.DataFrame = reduce(
    lambda left, right: left.join(
        right,
        on=["patientunitstayid", "time_offset"],
        how="full",
        coalesce=True,
        join_nulls=True
    ),
    # resampled_frames_to_merge_ts,
    df_list_ts,
)

df_joined_ts = df_list_ts[0].join(
    df_list_ts[1],
    on=["patientunitstayid", "time_offset"],
    # left_on = ["patientunitstayid", "time_offset"] ,
    # right_on = ["patientunitstayid", "time_offset"] ,
    how="full",
    # suffix="",
    # join_nulls=True,
    coalesce=True,
)


In [ ]:
# df_joined_ts.head().collect()
#df_joined_ts.collect()
df_joined_ts.head()

In [ ]:
reference_dataset.head().collect()

In [ ]:
testing_dataset_path = project_root_path / Path("intermediary_data/dataset/testing/merge_testing.parquet")

In [ ]:
df_joined_ts.sink_parquet(
    path=testing_dataset_path,
    compression="zstd",
    # compression_level=0,
)  # n